In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd

from slrrag.settings import (
    PANDERA_LAZY_VALIDATION,
    PATH_CSV_01_QUERY_RESULTS,
    NAME_COL_LANGUAGE_OPENALEX,
    SchemaPapers,
)
from slrrag.toolbox.toolbox_notebooks import (
    explode_and_filter_on_list,
    explode_and_groupby_topic,
    filter_on_list,
    group_by_col_ref,
    plot_histogram,
)

df_01_query_results = pd.read_csv(PATH_CSV_01_QUERY_RESULTS)


# Uncomment for demos
# df_papers = pd.read_csv(PATH_CSV_01_QUERY_RESULTS.parent / "demos" / "01_papers_query_results_demo.csv")

# Removed other reference papers papers not found
df_01_query_results = df_01_query_results[df_01_query_results["is_found"]]
df_01_query_results = SchemaPapers.validate(
    df_01_query_results, lazy=PANDERA_LAZY_VALIDATION
)
df_01_query_results

In [ ]:
df_01_query_results.query("is_reference")

In [ ]:
minimum_year_conference_paper = 2023
df_recent = df_01_query_results[
    (
        (df_01_query_results["type"] == "article")
        | (
            (df_01_query_results["type"] == "conference-paper")
            & (df_01_query_results["year"] >= minimum_year_conference_paper)
        )
        & (
            (df_01_query_results[NAME_COL_LANGUAGE_OPENALEX].isna())
            | (df_01_query_results[NAME_COL_LANGUAGE_OPENALEX] == "en")
        )
    )
]
print(len(df_01_query_results) - len(df_recent), "papers removed")
# df_recent = df_recent[
#     ~(df_recent["doi"].duplicated(keep="first")) | (df_recent["doi"].isna())
# ]  # Remove duplicates DOI
df_recent

In [ ]:
df_recent.query("is_reference")

In [ ]:
df_recent.groupby(["is_chosen_first_nra"]).size().sort_values(ascending=False)

# Topic analysis

## OpenAlex

#### Domain

In [ ]:
group_by_col_ref(df_recent, "domain_openalex", reference_colunm="is_chosen_first_nra")

In [ ]:
df_recent.dtypes

In [ ]:
black_list_domain_openalex = {
    "Health Sciences",
    "Life Sciences",
}
df_openalex_filtering_domain = filter_on_list(
    df_recent, black_list_domain_openalex, "fields_openalex", blacklist=True
)
group_by_col_ref(
    df_openalex_filtering_domain,
    "domain_openalex",
    reference_colunm="is_chosen_first_nra",
)

### Fields

In [ ]:
plot_histogram(
    df_openalex_filtering_domain,
    "fields_openalex",
    reference_colunm="is_chosen_first_nra",
)

In [ ]:
df_openalex_filtering_domain[
    df_openalex_filtering_domain["fields_openalex"] == "Environmental Science"
]

In [ ]:
group_by_col_ref(
    df_openalex_filtering_domain,
    "fields_openalex",
    reference_colunm="is_chosen_first_nra",
)

In [ ]:
black_list_fields_openalex = {
    "Agricultural and Biological Sciences",
    "Arts and Humanities",
    "Biochemistry, Genetics and Molecular Biology",
    "Chemical Engineering",
    "Chemistry",
    "Dentistry",
    "Health Professions",
    "Immunology and Microbiology",
    "Medicine",
    "Neuroscience",
    "Nursing",
    "Pharmacology, Toxicology and Pharmaceutics",
    "Psychology",
    "Business, Management and Accounting",
    "Materials Science",
    "Economics, Econometrics and Finance",
    "Physics and Astronomy",
}
df_openalex_filtering_fields = filter_on_list(
    df_recent, black_list_fields_openalex, "fields_openalex", blacklist=True
)
group_by_col_ref(
    df_openalex_filtering_fields,
    "fields_openalex",
    reference_colunm="is_chosen_first_nra",
).sort_values("fields_openalex")

### Subfields

In [ ]:
group_by_col_ref(
    df_openalex_filtering_fields,
    col_name="subfields_openalex",
    reference_colunm="is_chosen_first_nra",
).sort_values(["is_chosen_first_nra", "count"], ascending=False)

In [ ]:
blacklist_subfields_openalex = {"Biomedical Engineering", "Epidemiology", "Energy"}
df_openalex_filtering_subfields = filter_on_list(
    df_openalex_filtering_fields,
    blacklist_subfields_openalex,
    "subfields_openalex",
    blacklist=True,
)
group_by_col_ref(
    df_openalex_filtering_subfields,
    "subfields_openalex",
    reference_colunm="is_chosen_first_nra",
).sort_values(["is_chosen_first_nra", "count"], ascending=False)

In [ ]:
subfields = "Environmental Engineering"
df_openalex_filtering_subfields[
    df_openalex_filtering_subfields["subfields_openalex"] == subfields
].sort_values("relevance_openalex", ascending=False)[["title", "relevance_openalex"]]

In [ ]:
plot_histogram(
    df_openalex_filtering_subfields,
    "subfields_openalex",
    reference_colunm="is_chosen_first_nra",
    min_count_to_keep=2,
    height=750,
)

#### Topic

In [ ]:
group_by_col_ref(
    df_openalex_filtering_subfields,
    "topic_openalex",
    reference_colunm="is_chosen_first_nra",
).sort_values(["is_chosen_first_nra", "count"], ascending=False)

In [ ]:
black_list_openalex_topic = {
    "Arctic and Antarctic ice dynamics",
    "Fatigue and fracture mechanics",
    "Marine animal studies overview",
    "Spacecraft and Cryogenic Technologies",
    "Oceanographic and Atmospheric Processes",
    "Drilling and Well Engineering",
    "Fluid Dynamics and Vibration Analysis",
    "Engineering Structural Analysis Methods",
    "Structural Response to Dynamic Loads",
    "Fluid Dynamics Simulations and Interactions",
}
# Note : we skip lens filtering
df_openalex_filtering_topic = filter_on_list(
    df_openalex_filtering_subfields,
    black_list_openalex_topic,
    "topic_openalex",
    blacklist=True,
)
group_by_col_ref(
    df_openalex_filtering_topic,
    "topic_openalex",
    reference_colunm="is_chosen_first_nra",
).sort_values(["is_chosen_first_nra", "count"], ascending=False)

In [ ]:
df_openalex_filtering_subfields.query("is_chosen_first_nra").assign(
    in_blacklist=lambda x: x["topic_openalex"].isin(black_list_openalex_topic)
)  # .groupby("topic_openalex").size()

In [ ]:
plot_histogram(
    df_openalex_filtering_topic,
    "topic_openalex",
    reference_colunm="is_chosen_first_nra",
    min_count_to_keep=5,
    height=1000,
)

## Topic in dimensions

In [ ]:
explode_and_groupby_topic(
    df_openalex_filtering_topic,
    name_topic_col="topic_dimensions",
    separator=";",
    reference_colunm="is_chosen_first_nra",
)

In [ ]:
black_list_topic_dimensions = {
    "A01 Clinical Medicine",
    "A01 Clinical Medicine; A03 Allied Health Professions, Dentistry, Nursing and Pharmacy",
    "A01 Clinical Medicine; A04 Psychology, Psychiatry and Neuroscience",
    "A02 Public Health, Health Services and Primary Care",
    "A02 Public Health, Health Services and Primary Care; A03 Allied Health Professions, Dentistry, Nursing and Pharmacy",
    "A03 Allied Health Professions, Dentistry, Nursing and Pharmacy",
    "A04 Psychology, Psychiatry and Neuroscience",
    "A05 Biological Sciences",
    "A06 Agriculture, Veterinary and Food Science",
    "C23 Education",
    "C24 Sport and Exercise Sciences, Leisure and Tourism",
    "D25 Area Studies",
    "D26 Modern Languages and Linguistics",
    "D28 History",
    "D32 Art and Design: History, Practice and Theory",
    "D33 Music, Drama, Dance, Performing Arts, Film and Screen Studies",
    "D34 Communication, Cultural and Media Studies, Library and Information Management",
}
df_dimensions_filtering_topic = explode_and_filter_on_list(
    df_openalex_filtering_topic,
    black_list_input=black_list_topic_dimensions,
    col_name="topic_dimensions",
    seaparator=";",
)
explode_and_groupby_topic(
    df_dimensions_filtering_topic,
    "topic_dimensions",
    separator=";",
    reference_colunm="is_chosen_first_nra",
).sort_values(["is_chosen_first_nra", "count"], ascending=False)

In [ ]:
plot_histogram(
    df_dimensions_filtering_topic,
    "topic_dimensions",
    reference_colunm="is_chosen_first_nra",
    min_count_to_keep=1,
    height=500,
)

In [ ]:
df_dimensions_filtering_topic[df_dimensions_filtering_topic["is_chosen_first_nra"]]

# Scopus

### Complete scopus

In [ ]:
from slrrag.querier_scopus import complete_scopus_df

COMPLETE_SCOPUS_DF = False  # Turn it to true for scopus verification
# CAUTION : this takes a lot of time
if COMPLETE_SCOPUS_DF:
    df_scopus_completed = complete_scopus_df(df_dimensions_filtering_topic[50:100])
    plot_histogram(
        df_scopus_completed,
        col_name="subject_area",
        seperator="|",
        min_count_to_keep=2,
        height=500,
    )

### Keywords analysis

In [ ]:
df_dimensions_filtering_topic[~df_dimensions_filtering_topic["keywords_scopus"].isna()]

In [ ]:
plot_histogram(
    df_dimensions_filtering_topic,
    col_name="keywords_scopus",
    seperator="|",
    min_count_to_keep=5,
    height=500,
)

## Export to csv

In [ ]:
df_01_query_results

In [ ]:
from slrrag.settings import PATH_CSV_02_FILTERED_TOPIC

print("Total number of kept papers :", len(df_dimensions_filtering_topic))
print(
    "Number of reference papers:",
    len(df_dimensions_filtering_topic.query("is_reference")),
    "\n-----",
)

print(
    "Number of paper removed:",
    len(df_01_query_results) - len(df_dimensions_filtering_topic),
)
print(
    "Number of reference papers removed:",
    len(df_01_query_results.query("is_reference"))
    - len(df_dimensions_filtering_topic.query("is_reference")),
)
print(
    "Number of first NRA LR papers removed:",
    len(df_01_query_results.query("is_chosen_first_nra"))
    - len(df_dimensions_filtering_topic.query("is_chosen_first_nra")),
    "\n---",
)

print("Saving results to", PATH_CSV_02_FILTERED_TOPIC)
df_dimensions_filtering_topic.to_csv(PATH_CSV_02_FILTERED_TOPIC, index=False)

## External analysis

In [ ]:
df_dimensions_filtering_topic

In [ ]:
plot_histogram(
    df_dimensions_filtering_topic,
    col_name="subfields_openalex",
    min_count_to_keep=15,
    height=750,
    reference_colunm="is_chosen_first_nra",
)

In [ ]:
df_dimensions_filtering_topic.query("is_reference")

In [ ]:
from slrrag.toolbox import query_treemap

main_query = """
(
    (
        ("offshore") 
        AND 
        ("infrastructure" OR "construction" OR "installation" OR "windfarm" OR "wind farm" OR  "windmill" OR 
        "wind turbine" OR "wind park" OR "wind energy" OR "platform")
    )
    OR 
    ("flotel" OR "Aquaculture farm" OR "Artificial Island" OR "Synthetic Island")
)
AND
(
	( "collision" OR "allision" OR "near miss" OR "near-miss" OR "risk" OR "safety" OR "safe") AND 
    ( "navigate" OR "navigation" OR "navigational" OR "maritime" OR "shipping" OR "traffic")
	AND 
	("quantitative" OR "quantification" OR "probability" OR "probabilistic" OR "likelihood" 
        OR "frequency" OR "assess" OR "assessment" OR "predict" OR "predicting"
        OR "prediction" OR "model" OR "modeling" OR "modelling" OR "simulate"
        OR "simulation" OR "estimate" OR "estimation" OR "detect" OR "detection")
)
"""
tree = query_treemap.parse_query(main_query)
agg = query_treemap.aggregate_by_node(
    df=df_dimensions_filtering_topic,
    tree=tree,
    columns=["title", "abstract"],
    binary_columns=["is_reference"],
    stemming=False,
).sort_values("count", ascending=False)
fig = query_treemap.treemap(agg, width=1250, height=750)
fig

## Convert to markdown for Obsidian

In [ ]:
from pathlib import Path

from slrrag.convertmarkdown.paper_review_tools import write_series_papers_into_markdown
from slrrag.settings import NAME_COL_RELEVANCE_OPENALEX, NAME_COL_RELEVANCE_ORDER_SCOPUS


input_csv = PATH_CSV_02_FILTERED_TOPIC
output_dir = Path(
    "/mnt/c/Users/vlitoux/OneDrive - Delft University of Technology/Documents/Analyse_papers/all_papers_query_19"
)

# Read the CSV
df = pd.read_csv(input_csv)
# Sort if relevance in columns, first openalex and then scopus
# The boolean indicates if it is ascending or not
dict_sort_relevance = {
    NAME_COL_RELEVANCE_OPENALEX: False,
    NAME_COL_RELEVANCE_ORDER_SCOPUS: True,
}
list_col_sort = [x for x in dict_sort_relevance.keys() if x in df.columns]
list_ascending = [
    dict_sort_relevance[x] for x in dict_sort_relevance.keys() if x in df.columns
]
if len(list_col_sort) > 0:
    df = df.sort_values(
        by=list_col_sort, ascending=list_ascending
    )  # Create markdown files
print(f"Processing {len(df)} papers from {input_csv}...")
created_files = write_series_papers_into_markdown(df, output_dir=output_dir)

print(f"\nDone! Created {len(created_files)} markdown files in {output_dir}")